In [7]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [8]:
import torch

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MaskedBatchNorm(nn.BatchNorm1d):
    '''
    computes batch norm statistics from just the non-zero padded tracks
    '''

    def forward(self, x, mask=None):
        '''
            x: [B*M, V*C, T]
        '''

        if mask is None or not self.training: # dont update statistics if not training
            return super().forward(x)

        real = x[mask]
        batch_mean = real.mean(dim=(0,2))
        batch_var = real.var(dim=(0,2), unbiased=False)

        with torch.no_grad():
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var

        # set training to false because we've already computed running mean/var
        return F.batch_norm(x, batch_mean, batch_var, self.weight, self.bias, training=False, eps=self.eps)